# MCP长期订正的合成示例

现场测量通常只覆盖有限时段，风资源评估需要判断这段测量在长期气候中的位置。Measure-Correlate-Predict（MCP）利用现场序列与长期参考资料的重叠时段建立关系，再将关系应用到更长的参考序列。

这个Notebook使用合成数据。完整“真实场址序列”只用于检验示例，在实际项目中通常并不可见。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

months = np.arange(20 * 12)
season = 1.3 * np.sin(2 * np.pi * months / 12.0)
reference = 6.5 + season + rng.normal(0, 0.8, size=months.size)

site_true = 1.0 + 0.88 * reference + rng.normal(0, 0.45, size=months.size)

overlap_start = 8 * 12
overlap_end = 10 * 12
mask = (months >= overlap_start) & (months < overlap_end)

coef = np.polyfit(reference[mask], site_true[mask], deg=1)
site_reconstructed = np.polyval(coef, reference)

print(f"fitted slope={coef[0]:.3f}")
print(f"fitted intercept={coef[1]:.3f}")
print(f"true long-term mean={site_true.mean():.3f} m/s")
print(f"reconstructed long-term mean={site_reconstructed.mean():.3f} m/s")
print(f"mean bias={site_reconstructed.mean() - site_true.mean():+.3f} m/s")

这个例子只使用24个月重叠资料。拟合误差同时受有限样本和场址—参考序列关系的随机波动影响。

In [ ]:
years = 2000 + months / 12.0

plt.figure(figsize=(10, 4.5))
plt.plot(years, site_true, label="synthetic site truth", linewidth=1)
plt.plot(years, site_reconstructed, label="MCP reconstruction", linewidth=1)
plt.axvspan(
    2000 + overlap_start / 12.0,
    2000 + overlap_end / 12.0,
    alpha=0.15,
    label="overlap period"
)
plt.xlabel("Year")
plt.ylabel("Wind speed (m/s)")
plt.legend()
plt.tight_layout()
plt.show()

## 重叠时段长度

同一条长期参考序列，用12、24和60个月重叠数据拟合，得到的参数会有差异。

In [ ]:
for length in [12, 24, 60]:
    start = overlap_start
    end = start + length
    m = (months >= start) & (months < end)
    a, b = np.polyfit(reference[m], site_true[m], deg=1)
    pred = a * reference + b
    rmse = np.sqrt(np.mean((pred - site_true) ** 2))
    print(
        f"{length:>2} months | slope={a:.3f} | "
        f"intercept={b:.3f} | long-term RMSE={rmse:.3f} m/s"
    )

重叠时间更长通常能覆盖更多季节和年际状态，但仍需处理参考资料偏差、场址变化和统计关系失稳。实际MCP也会采用分风向、分季节、多变量或非线性方法。

参考：

- IEC 61400-15-1:2025  
  https://webstore.iec.ch/en/publication/29169
- Zhou & Esau (2026), https://doi.org/10.5194/wes-11-217-2026
- Borowski et al. (2026, preprint), https://doi.org/10.5194/wes-2026-129